# Two-window Cell Painting analysis (48 h vs 8 days)

Reproduces the analyses and data figures of the manuscript
"A stability-weighted combined map for Cell Painting across regular and prolonged incubation".

Pipeline: per-well profiles (already normalized to DMSO and feature-selected)
-> one shared PCA for all wells -> within-iteration replicate stability -> window selection
-> one median profile per compound and iteration -> kNN graph + Leiden (100 seeds)
-> silhouette score and cross-iteration agreement -> random-window null model
-> parameter grid for all datasets -> reproducibility of the window choice -> figures and tables.

Input: per-well feature table (CSV) with columns Metadata_Name, Metadata_Source
(sh_1, sh_2 = 48 h iterations 1 and 2; l_1, l_2 = 8 days iterations 1 and 2) and numeric features.
Outputs are written to OUT_DIR.

In [ ]:
import os, sys, json, time, itertools, collections, warnings
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, cohen_kappa_score
from scipy.spatial.distance import pdist
from scipy.stats import binomtest, wilcoxon, mannwhitneyu, fisher_exact
from statsmodels.stats.contingency_tables import cochrans_q
from statsmodels.stats.proportion import proportion_confint

warnings.filterwarnings('ignore')
import logging; logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
sc.settings.verbosity = 0

## Settings

In [ ]:
CONFIG = {
    'FILE': 'INPUT',                 # path to the per-well CSV table
    'OUT_DIR': 'Result',             # output folder
    'MIN_LONG_ITERS': 2,             # keep compounds with both 8-day iterations
    # main analysis configuration (fixed for all three datasets)
    'N_COMPS': 55, 'METRIC': 'correlation', 'K': 5, 'RES': 0.6,
    'PCA_RANDOM_STATE': 42, 'UMAP_RANDOM_STATE': 42,
    'N_SEEDS': 100,                  # Leiden random seeds
    'N_NULL': 1000,                  # random window assignments
    'NULL_SEEDS': 10,                # Leiden seeds per random assignment
    'NULL_RNG': 2026,                # random generator for window assignment
    # parameter grid (evaluated for all three datasets)
    'GRID_N_COMPS': list(range(50, 71, 5)),
    'GRID_METRICS': ['euclidean', 'cosine', 'correlation'],
    'GRID_K': list(range(5, 31, 5)),
    'GRID_RES': [round(x, 1) for x in np.arange(0.2, 3.01, 0.2)],
    'FC_CLUSTER_RANGE': (5, 9),      # cluster-number range for the control selection criterion
    'RUN_GRID': True, 'RUN_NULL': True, 'MAKE_FIGURES': True,
    'RUN_CONTROL_CONFIG': True,      # repeat seeds and null model for the configuration chosen without the Combined dataset
}
DATASETS = ['short', 'long', 'comb']
NAMES = {'short': '48 h', 'long': '8 days', 'comb': 'Combined'}

## Data loading and shared PCA

In [ ]:
def load_dataset(filename):
    df = pd.read_csv(filename, index_col=None)
    meta_cols = [c for c in df.columns if c.lower().startswith('metadata_')]
    feat_cols = [c for c in df.columns if c not in meta_cols and np.issubdtype(df[c].dtype, np.number)]
    return df, meta_cols, feat_cols


def filter_by_long_iterations(df, min_long_iters=2):
    """Keep compounds with >= min_long_iters independent 8-day iterations."""
    is_long = df['Metadata_Source'].astype(str).str.startswith('l')
    n_it = df[is_long].groupby('Metadata_Name')['Metadata_Source'].nunique()
    keep = n_it[n_it >= min_long_iters].index
    dropped = sorted(set(df['Metadata_Name']) - set(keep))
    print(f'Compounds: {df.Metadata_Name.nunique()} -> {len(keep)} (excluded: {dropped})')
    return df[df['Metadata_Name'].isin(keep)].reset_index(drop=True)


def annotate(df):
    meta = df[['Metadata_Name', 'Metadata_Source']].copy()
    meta['window'] = np.where(meta['Metadata_Source'].str.startswith('sh'), 'short', 'long')
    meta['iter'] = meta['Metadata_Source'].str.split('_').str[-1].astype(int)
    return meta


def shared_pca(X, n_comps, random_state=42, svd_solver='auto'):
    """One StandardScaler + PCA fitted to ALL wells of both windows and both iterations."""
    Z = StandardScaler().fit_transform(X)
    pca = PCA(n_components=n_comps, random_state=random_state, svd_solver=svd_solver).fit(Z)
    return pca.transform(Z), pca.explained_variance_ratio_.sum()

## Window selection (within-iteration replicate stability)

In [ ]:
def replicate_distance(meta, P, metric='correlation', mode='within'):
    """Mean pairwise distance between replicate wells of a compound in each window.
    mode='within': only pairs of wells from the same iteration (used in the paper);
    mode='iter1' / 'iter2': wells of one iteration only (reproducibility of the choice)."""
    out = {}
    for (c, w), g in meta.groupby(['Metadata_Name', 'window']):
        if mode == 'within':
            d = [pdist(P[gi.index.values], metric) for _, gi in g.groupby('iter') if len(gi) > 1]
            d = np.concatenate(d) if d else np.array([])
        else:
            gi = g[g['iter'] == int(mode[-1])]
            d = pdist(P[gi.index.values], metric) if len(gi) > 1 else np.array([])
        out[(c, w)] = d.mean() if len(d) else np.nan
    return pd.Series(out).unstack()   # columns: long, short


def select_windows(D):
    """For each compound, the window with the smaller replicate distance."""
    return {c: ('long' if r['long'] < r['short'] else 'short') for c, r in D.iterrows()}


def build_profiles(meta, P, choice):
    """One median PC profile per compound and iteration from the assigned window."""
    keys, rows = [], []
    for c in sorted(choice):
        w = choice[c]
        for it in (1, 2):
            m = ((meta['Metadata_Name'] == c) & (meta['window'] == w) & (meta['iter'] == it)).values
            if m.sum() == 0:
                continue
            rows.append(np.median(P[m], axis=0)); keys.append((c, it, w))
    return pd.DataFrame(keys, columns=['compound', 'iter', 'window']), np.vstack(rows)

## Clustering and metrics

In [ ]:
def knn_graph(Y, k, metric):
    a = ad.AnnData(Y.astype(np.float32)); a.obsm['X_pca'] = Y
    sc.pp.neighbors(a, n_neighbors=k, use_rep='X_pca', metric=metric)
    return a


def leiden(a, res, seed):
    sc.tl.leiden(a, resolution=res, random_state=seed, flavor='leidenalg', directed=False, key_added='l')
    return a.obs['l'].astype(int).values


def evaluate(keys, Y, lab, metric):
    sil = silhouette_score(Y, lab, metric=metric) if len(set(lab)) > 1 else np.nan
    k = keys.copy(); k['lab'] = lab
    disc = k.groupby('compound')['lab'].nunique() > 1
    return sil, int(disc.sum()), len(set(lab)), set(disc[disc].index)


def canonical(lab):
    order = sorted(set(lab), key=lambda c: (-np.sum(lab == c), np.where(lab == c)[0][0]))
    m = {c: i for i, c in enumerate(order)}
    return tuple(m[c] for c in lab)


def seed_sweep(keys, Y, cfg, seeds):
    a = knn_graph(Y, cfg['K'], cfg['METRIC'])
    rows, parts = [], []
    for s in seeds:
        lab = leiden(a, cfg['RES'], s)
        sil, d, nc, _ = evaluate(keys, Y, lab, cfg['METRIC'])
        rows.append((s, sil, d, nc)); parts.append(canonical(lab))
    modal, freq = collections.Counter(parts).most_common(1)[0]
    return pd.DataFrame(rows, columns=['seed', 'sil', 'disc', 'ncl']), np.array(modal), freq, a


def mean_over_seeds(meta, P, choice, cfg, seeds):
    keys, Y = build_profiles(meta, P, choice)
    a = knn_graph(Y, cfg['K'], cfg['METRIC'])
    r = [evaluate(keys, Y, leiden(a, cfg['RES'], s), cfg['METRIC'])[:2] for s in seeds]
    return float(np.mean([x[0] for x in r])), float(np.mean([x[1] for x in r]))

## Main analysis

In [ ]:
def main(cfg=CONFIG):
    t0 = time.time()
    out = cfg['OUT_DIR']; os.makedirs(out, exist_ok=True)
    df, meta_cols, feat_cols = load_dataset(cfg['FILE'])
    df = filter_by_long_iterations(df, cfg['MIN_LONG_ITERS'])
    meta = annotate(df); X = df[feat_cols].values.astype(float)
    print(f'Wells: {len(df)}; features: {len(feat_cols)}')
    P, var = shared_pca(X, cfg['N_COMPS'], cfg['PCA_RANDOM_STATE'])
    print(f'Shared PCA: {cfg["N_COMPS"]} components, {100*var:.1f}% of variance')
    comps = sorted(meta['Metadata_Name'].unique())
    R = {'n_compounds': len(comps), 'n_wells': len(df), 'n_features': len(feat_cols), 'pca_variance': var}

    # --- window selection
    Dw = replicate_distance(meta, P, cfg['METRIC'], 'within')
    choice = select_windows(Dw)
    delta = ((Dw['long'] - Dw['short']).abs() / ((Dw['long'] + Dw['short']) / 2))
    sets = {'short': {c: 'short' for c in comps}, 'long': {c: 'long' for c in comps}, 'comb': choice}
    R['window_counts'] = collections.Counter(choice.values())
    print('Selected windows:', dict(R['window_counts']))

    # --- 100 Leiden seeds, modal partitions, UMAP
    seeds = range(cfg['N_SEEDS']); S = {}; modal = {}
    for n, ch in sets.items():
        keys, Y = build_profiles(meta, P, ch)
        sw, mlab, freq, a = seed_sweep(keys, Y, cfg, seeds)
        sw['set'] = n; S[n] = sw
        sil, d, nc, discset = evaluate(keys, Y, mlab, cfg['METRIC'])
        sc.tl.umap(a, random_state=cfg['UMAP_RANDOM_STATE'])
        k = keys.copy(); k['cluster'] = mlab; k['umap1'] = a.obsm['X_umap'][:, 0]; k['umap2'] = a.obsm['X_umap'][:, 1]
        modal[n] = k; k.to_csv(os.path.join(out, f'modal_partition_{n}.csv'), index=False)
        R[n] = {'modal_freq': freq, 'modal_silhouette': sil, 'modal_discordant': d, 'modal_clusters': nc,
                'median_silhouette': sw.sil.median(), 'silhouette_2.5_97.5': list(sw.sil.quantile([.025, .975])),
                'median_discordant': sw.disc.median(), 'discordant_2.5_97.5': list(sw.disc.quantile([.025, .975]))}
        print(f'{NAMES[n]:9s} modal: silhouette {sil:.4f}, discordant {d}, clusters {nc} (in {freq}/{len(seeds)} seeds); '
              f'median silhouette {sw.sil.median():.3f}, median discordant {sw.disc.median():.0f}')
    seeds_df = pd.concat(S.values()); seeds_df.to_csv(os.path.join(out, 'leiden_seeds.csv'), index=False)
    W = seeds_df.pivot(index='seed', columns='set', values=['sil', 'disc'])
    R['seed_frac_sil_better'] = float((W.sil.comb > W.sil[['short', 'long']].max(1)).mean())
    R['seed_frac_disc_better'] = float((W.disc.comb < W.disc[['short', 'long']].min(1)).mean())
    print(f'Combined better than both windows: silhouette in {100*R["seed_frac_sil_better"]:.0f}% of seeds, '
          f'discordance in {100*R["seed_frac_disc_better"]:.0f}%')

    # --- discordance tests on modal partitions (Cochran's Q, exact McNemar with Holm)
    M = pd.DataFrame({n: (modal[n].groupby('compound')['cluster'].nunique() > 1).astype(int) for n in DATASETS})
    q = cochrans_q(M.values); R['cochran_Q'] = (q.statistic, q.pvalue)
    raw = {}
    for a_, b_ in [('short', 'comb'), ('long', 'comb'), ('short', 'long')]:
        b1 = int(((M[a_] == 1) & (M[b_] == 0)).sum()); c1 = int(((M[a_] == 0) & (M[b_] == 1)).sum())
        raw[f'{a_}_vs_{b_}'] = (b1, c1, binomtest(min(b1, c1), b1 + c1, 0.5).pvalue)
    order = sorted(raw, key=lambda x: raw[x][2]); prev = 0; holm = {}
    for i, kname in enumerate(order):
        prev = max(prev, min(1, (len(order) - i) * raw[kname][2])); holm[kname] = prev
    R['mcnemar'] = {kname: {'only_first_discordant': v[0], 'only_second_discordant': v[1], 'p': v[2], 'p_holm': holm[kname]} for kname, v in raw.items()}
    print(f"Cochran's Q = {q.statistic:.2f}, p = {q.pvalue:.3f}; McNemar:",
          {kname: (round(v[2], 4), round(holm[kname], 3)) for kname, v in raw.items()})

    # --- random-window null model and reverse rule
    if cfg['RUN_NULL']:
        nseeds = range(cfg['NULL_SEEDS'])
        obs = mean_over_seeds(meta, P, choice, cfg, nseeds)
        rev = mean_over_seeds(meta, P, {c: ('short' if w == 'long' else 'long') for c, w in choice.items()}, cfg, nseeds)
        rng = np.random.default_rng(cfg['NULL_RNG']); null = []
        for b in range(cfg['N_NULL']):
            ch = {c: rng.choice(['short', 'long']) for c in comps}
            null.append(mean_over_seeds(meta, P, ch, cfg, nseeds))
            if (b + 1) % 100 == 0: print(f'  null model: {b+1}/{cfg["N_NULL"]}')
        null = np.array(null); np.save(os.path.join(out, 'null_model.npy'), null)
        R['null'] = {'observed': obs, 'reverse': rev,
                     'null_median': list(np.median(null, 0)), 'null_2.5': list(np.quantile(null, .025, 0)), 'null_97.5': list(np.quantile(null, .975, 0)),
                     'p_silhouette': float((np.sum(null[:, 0] >= obs[0]) + 1) / (len(null) + 1)),
                     'p_discordant': float((np.sum(null[:, 1] <= obs[1]) + 1) / (len(null) + 1)),
                     'reverse_worse_than_frac': [float((null[:, 0] > rev[0]).mean()), float((null[:, 1] < rev[1]).mean())]}
        print(f'Null model: observed silhouette {obs[0]:.3f}, discordant {obs[1]:.1f}; '
              f'random median {np.median(null[:,0]):.3f} / {np.median(null[:,1]):.1f}; '
              f'p = {R["null"]["p_silhouette"]:.3f} / {R["null"]["p_discordant"]:.3f}; reverse rule {rev[0]:.3f} / {rev[1]:.1f}')

    # --- reproducibility of the window choice
    d1 = replicate_distance(meta, P, cfg['METRIC'], 'iter1'); d2 = replicate_distance(meta, P, cfg['METRIC'], 'iter2')
    ok = d1.notna().all(1) & d2.notna().all(1)
    c1 = pd.Series(select_windows(d1[ok])); c2 = pd.Series(select_windows(d2[ok])); agree = (c1 == c2)
    dl = delta[agree.index]; terc = pd.qcut(dl, 3, labels=['lowest', 'middle', 'highest'])
    R['window_choice'] = {'n': int(ok.sum()), 'excluded': list(ok.index[~ok]), 'agreement': float(agree.mean()),
                          'kappa': float(cohen_kappa_score(c1, c2)),
                          'tertiles': {t: [int(agree[terc == t].sum()), int((terc == t).sum())] for t in ['lowest', 'middle', 'highest']},
                          'median_delta_identical_different': [float(dl[agree].median()), float(dl[~agree].median())],
                          'mannwhitney_p': float(mannwhitneyu(dl[agree], dl[~agree]).pvalue)}
    print('Window choice reproducibility:', {k_: v for k_, v in R['window_choice'].items() if k_ != 'excluded'})

    # --- temporal classes and their relation to delta
    kc = modal['comb']; mixed = [c for c, g in kc.groupby('cluster') if g['window'].nunique() > 1]
    cls = kc.groupby('compound').apply(lambda g: 'Convergent' if g['cluster'].isin(mixed).any()
                                       else ('48 h-stable' if g['window'].iloc[0] == 'short' else '8-day-stable'))
    t53 = pd.qcut(delta[cls.index], 3, labels=['lowest', 'middle', 'highest'])
    conv = cls == 'Convergent'
    R['classes'] = dict(cls.value_counts())
    R['convergent_lowest_delta'] = {'n_in_lowest': int((conv & (t53 == 'lowest')).sum()), 'n_convergent': int(conv.sum()),
                                    'fisher_p': float(fisher_exact(pd.crosstab(conv, t53 == 'lowest').values)[1])}
    print('Classes:', R['classes'], R['convergent_lowest_delta'])

    # --- tables
    tab = pd.DataFrame({'d_48h': Dw['short'], 'd_8d': Dw['long'], 'selected': pd.Series(choice), 'delta': delta,
                        'choice_iter1': c1, 'choice_iter2': c2, 'class': cls})
    tab.to_csv(os.path.join(out, 'window_selection_per_compound.csv'))
    for n in DATASETS:
        t = modal[n].pivot(index='compound', columns='iter', values='cluster')
        t.columns = ['cluster_iter1', 'cluster_iter2']; t['concordant'] = t.cluster_iter1 == t.cluster_iter2
        if n == 'comb': t['selected_window'] = pd.Series(choice); t['class'] = cls
        t.sort_values(['cluster_iter1', 'cluster_iter2']).to_csv(os.path.join(out, f'cluster_membership_{n}.csv'))

    # --- parameter grid for all three datasets
    if cfg['RUN_GRID']:
        G = run_grid_search(meta, X, comps, cfg)
        G.to_csv(os.path.join(out, 'grid_search_all_datasets.csv'), index=False)
        R['grid'] = summarize_grid(G, cfg)
        print('Grid:', R['grid'])
        if cfg['RUN_CONTROL_CONFIG']:
            R['control_config'] = control_config_analysis(meta, X, comps, cfg, R['grid']['control_selection_best']['config'], out)

    if cfg['MAKE_FIGURES']:
        make_figures(out, modal, seeds_df, R, (agree, dl, terc, c1, c2, cls),
                     G if cfg['RUN_GRID'] else None, null if cfg['RUN_NULL'] else None)

    with open(os.path.join(out, 'results_summary.json'), 'w') as f:
        json.dump(R, f, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
    print(f'Done in {time.time()-t0:.0f} s. Results in {out}')
    return R

## Parameter grid (all three datasets) and control selection

In [ ]:
def run_grid_search(meta, X, comps, cfg):
    Pfull, _ = shared_pca(X, max(cfg['GRID_N_COMPS']), cfg['PCA_RANDOM_STATE'], svd_solver='full')
    rows = []
    for npc, metric in itertools.product(cfg['GRID_N_COMPS'], cfg['GRID_METRICS']):
        P = Pfull[:, :npc]
        ch = select_windows(replicate_distance(meta, P, metric, 'within'))
        prof = {n: build_profiles(meta, P, s) for n, s in
                {'short': {c: 'short' for c in comps}, 'long': {c: 'long' for c in comps}, 'comb': ch}.items()}
        for k in cfg['GRID_K']:
            for n, (keys, Y) in prof.items():
                a = knn_graph(Y, k, metric)
                for res in cfg['GRID_RES']:
                    lab = leiden(a, res, 0)
                    sil, d, nc, _ = evaluate(keys, Y, lab, metric)
                    rows.append((npc, metric, k, res, n, sil, d, nc))
        print(f'  grid: {npc} PCs, {metric} done')
    return pd.DataFrame(rows, columns=['n_comps', 'metric', 'k', 'res', 'set', 'sil', 'disc', 'ncl'])


def summarize_grid(G, cfg):
    W = G.pivot_table(index=['n_comps', 'metric', 'k', 'res'], columns='set', values=['sil', 'disc', 'ncl']).dropna()
    W = W[W.ncl.min(1) >= 2]
    best_s = W.sil[['short', 'long']].max(1); best_d = W.disc[['short', 'long']].min(1)
    lo, hi = cfg['FC_CLUSTER_RANGE']
    g = W[(W.ncl[['short', 'long']].min(1) >= lo) & (W.ncl[['short', 'long']].max(1) <= hi)]
    Fc = (g.sil.short + g.sil.long) / 2 - 0.02 * (g.disc.short + g.disc.long) / 2
    top = g.loc[Fc.sort_values(ascending=False).index[:5]]
    best = Fc.idxmax(); rb = g.loc[best]
    return {'n_valid': int(len(W)),
            'frac_sil_better': float((W.sil.comb > best_s).mean()), 'median_sil_gain': float((W.sil.comb - best_s).median()),
            'wilcoxon_sil_p': float(wilcoxon(W.sil.comb, best_s).pvalue),
            'frac_disc_better': float((W.disc.comb < best_d).mean()), 'median_disc_change': float((W.disc.comb - best_d).median()),
            'wilcoxon_disc_p': float(wilcoxon(W.disc.comb, best_d).pvalue),
            'frac_sil_better_by_metric': {m: float((W.sil.comb.xs(m, level='metric') > best_s.xs(m, level='metric')).mean()) for m in cfg['GRID_METRICS']},
            'frac_disc_better_by_metric': {m: float((W.disc.comb.xs(m, level='metric') < best_d.xs(m, level='metric')).mean()) for m in cfg['GRID_METRICS']},
            'control_selection_best': {'config': list(map(str, best)), 'silhouette_48h_8d_comb': [rb.sil.short, rb.sil.long, rb.sil.comb],
                                       'discordant_48h_8d_comb': [rb.disc.short, rb.disc.long, rb.disc.comb]},
            'control_selection_top5_comb_best': bool(((top.sil.comb > top.sil[['short', 'long']].max(1)) &
                                                      (top.disc.comb < top.disc[['short', 'long']].min(1))).all())}

def control_config_analysis(meta, X, comps, cfg, best, out):
    """Repeat the seed analysis and the random-window null model for the configuration
    selected by the control criterion F_c, which does not use the Combined dataset
    (Supplementary Note 1)."""
    c2 = dict(cfg); c2['N_COMPS'], c2['METRIC'], c2['K'], c2['RES'] = int(best[0]), best[1], int(best[2]), float(best[3])
    P2, var2 = shared_pca(X, c2['N_COMPS'], c2['PCA_RANDOM_STATE'])
    choice2 = select_windows(replicate_distance(meta, P2, c2['METRIC'], 'within'))
    sets = {'short': {c: 'short' for c in comps}, 'long': {c: 'long' for c in comps}, 'comb': choice2}
    S = {}
    for n, ch in sets.items():
        keys, Y = build_profiles(meta, P2, ch)
        sw, _, _, _ = seed_sweep(keys, Y, c2, range(c2['N_SEEDS'])); sw['set'] = n; S[n] = sw
    sd = pd.concat(S.values()); sd.to_csv(os.path.join(out, 'control_config_leiden_seeds.csv'), index=False)
    W = sd.pivot(index='seed', columns='set', values=['sil', 'disc'])
    res = {'config': [c2['N_COMPS'], c2['METRIC'], c2['K'], c2['RES']], 'window_counts': dict(collections.Counter(choice2.values())),
           'median_silhouette': {n: float(S[n].sil.median()) for n in DATASETS},
           'median_discordant': {n: float(S[n].disc.median()) for n in DATASETS},
           'seed_frac_sil_better': float((W.sil.comb > W.sil[['short', 'long']].max(1)).mean()),
           'seed_frac_disc_better': float((W.disc.comb < W.disc[['short', 'long']].min(1)).mean())}
    nseeds = range(c2['NULL_SEEDS'])
    obs = mean_over_seeds(meta, P2, choice2, c2, nseeds)
    rev = mean_over_seeds(meta, P2, {c: ('short' if w == 'long' else 'long') for c, w in choice2.items()}, c2, nseeds)
    rng = np.random.default_rng(c2['NULL_RNG']); null = []
    for b in range(c2['N_NULL']):
        ch = {c: rng.choice(['short', 'long']) for c in comps}
        null.append(mean_over_seeds(meta, P2, ch, c2, nseeds))
    null = np.array(null); np.save(os.path.join(out, 'control_config_null_model.npy'), null)
    res['null'] = {'observed': obs, 'reverse': rev, 'null_median': list(np.median(null, 0)),
                   'null_2.5': list(np.quantile(null, .025, 0)), 'null_97.5': list(np.quantile(null, .975, 0)),
                   'p_silhouette': float((np.sum(null[:, 0] >= obs[0]) + 1) / (len(null) + 1)),
                   'p_discordant': float((np.sum(null[:, 1] <= obs[1]) + 1) / (len(null) + 1))}
    print('Control configuration', res['config'], {k: v for k, v in res.items() if k != 'config'})
    return res

## Figures (Figs 2-6 and Supplementary Fig. S1)

In [ ]:
plt.rcParams.update({'font.family': ['Arial', 'Liberation Sans', 'DejaVu Sans'], 'font.size': 8.5, 'mathtext.default': 'regular',
                     'axes.spines.top': False, 'axes.spines.right': False, 'axes.edgecolor': '#8a8985',
                     'axes.labelcolor': '#52514e', 'xtick.color': '#52514e', 'ytick.color': '#52514e'})
PAL = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#4a3aa7']
C48, C8, CC, GRAY, INK, INK2 = '#2a78d6', '#eb6834', '#1baf7a', '#c3c2b7', '#0b0b0b', '#52514e'
nice = lambda s: s.replace('_', ' ').replace('lenalidomid', 'lenalidomide').replace('cyclosporin a', 'cyclosporin A')


def fig_umap(k, n, fname, out, highlight=()):
    from adjustText import adjust_text
    disc = set(k.groupby('compound').cluster.nunique().loc[lambda s: s > 1].index)
    fig, ax = plt.subplots(figsize=(6.3, 5.6))
    for c, g in k.groupby('compound'):
        if len(g) == 2 and c in disc:
            ax.plot(g.umap1, g.umap2, color='#8a8985', lw=1.0, ls=(0, (3, 2)), zorder=1)
    for cl, g in k.groupby('cluster'):
        for w, m in [('short', 'o'), ('long', '^')]:
            gg = g[g.window == w]
            if len(gg):
                isd = gg.compound.isin(disc)
                ax.scatter(gg.umap1, gg.umap2, s=46, marker=m, c=PAL[cl % len(PAL)], edgecolors=np.where(isd, INK, '#fcfcfb'),
                           linewidths=np.where(isd, 1.3, 1.0), zorder=3)
    texts = []
    for cl, g in k.groupby('cluster'):
        texts.append(ax.text(g.umap1.median(), g.umap2.max(), f'C{cl}', fontsize=12, fontweight='bold', color=PAL[cl % len(PAL)],
                             ha='center', va='bottom', zorder=5, path_effects=[pe.withStroke(linewidth=3, foreground='white')]))
    for c in sorted(disc) + [h for h in highlight if h not in disc]:
        r = k[k.compound == c].sort_values('iter').iloc[0]
        texts.append(ax.text(r.umap1, r.umap2, nice(c), fontsize=7, color=INK, fontweight='bold', zorder=6,
                             path_effects=[pe.withStroke(linewidth=2.2, foreground='white')]))
    adjust_text(texts, ax=ax, x=k.umap1.values, y=k.umap2.values, arrowprops=dict(arrowstyle='-', color='#8a8985', lw=1.0),
                expand=(1.5, 1.8), force_text=(0.6, 0.9), force_static=(0.3, 0.5))
    ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2'); ax.set_xticks([]); ax.set_yticks([])
    h = [Line2D([], [], marker='o', ls='', mfc='#b0afa8', mec='white', ms=7, label='profile, 48 h window'),
         Line2D([], [], marker='^', ls='', mfc='#b0afa8', mec='white', ms=7, label='profile, 8-day window'),
         Line2D([], [], marker='o', ls='', mfc='white', mec=INK, mew=1.3, ms=7, label='discordant between iterations'),
         Line2D([], [], color='#8a8985', lw=1.0, ls=(0, (3, 2)), label='discordant compound: iteration 1 ↔ 2')]
    if n != 'comb': h = [h[0] if n == 'short' else h[1]] + h[2:]
    ax.legend(handles=h, loc='lower left', bbox_to_anchor=(0, -0.2), ncol=2, frameon=False, fontsize=7.5)
    fig.tight_layout(); fig.savefig(os.path.join(out, fname + '.png'), dpi=300); fig.savefig(os.path.join(out, fname + '.pdf')); plt.close(fig)


def make_figures(out, modal, seeds_df, R, wc, G, null):
    fig_umap(modal['short'], 'short', 'Fig2_UMAP_48h', out)
    fig_umap(modal['long'], 'long', 'Fig3_UMAP_8d', out)
    fig_umap(modal['comb'], 'comb', 'Fig4_UMAP_Combined', out,
             highlight=('palbociclib', 'etoposide', 'ceralasertib', 'alpelisib', 'ku-0063794'))
    panel = lambda ax, L: ax.text(-0.14, 1.06, L, transform=ax.transAxes, fontsize=11, fontweight='bold', color=INK)
    # Fig 5: null model
    if null is not None:
        obs, rev = R['null']['observed'], R['null']['reverse']
        fig, axs = plt.subplots(1, 2, figsize=(7.2, 2.9))
        for j, (ax, xl, bins) in enumerate(zip(axs, ['Silhouette score, Combined', 'Discordant compounds (of 53), Combined'],
                                              [35, np.arange(3, 21, 0.5)])):
            ax.hist(null[:, j], bins=bins, color=GRAY, edgecolor='white', linewidth=0.5)
            p = R['null']['p_silhouette'] if j == 0 else R['null']['p_discordant']
            for v, lab_, col, ls in [(obs[j], f'stability-based\n(p = {p:.3g})', CC, '-'), (rev[j], 'reverse rule', INK2, ':')]:
                ax.axvline(v, color=col, lw=2, ls=ls); ax.text(v, ax.get_ylim()[1] * 0.97, ' ' + lab_, color=INK if col == CC else INK2, va='top', fontsize=7.5)
            ax.set_xlabel(xl); ax.set_ylabel('Random window assignments'); panel(ax, 'ab'[j])
        axs[0].set_title(f'Random window assignment (n = {len(null):,})', fontsize=8.5, loc='left', color=INK)
        fig.tight_layout(); fig.savefig(os.path.join(out, 'Fig5_null.png'), dpi=300); fig.savefig(os.path.join(out, 'Fig5_null.pdf')); plt.close(fig)
    # Fig 6: reproducibility of the window choice
    agree, dl, terc, c1, c2, cls = wc
    fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.2), gridspec_kw={'width_ratios': [1, 1.1]})
    ax = axs[0]; qs = ['lowest', 'middle', 'highest']; cols = [GRAY, '#86b6ef', C48]
    for i, qn in enumerate(qs):
        k_, n_ = int(agree[terc == qn].sum()), int((terc == qn).sum()); lo, hi = proportion_confint(k_, n_, method='wilson')
        ax.plot([i, i], [lo, hi], color=INK2, lw=1.2, zorder=2)
        ax.scatter([i], [k_ / n_], s=60, color=cols[i], edgecolors=INK, linewidths=0.8, zorder=3)
        ax.text(i + 0.12, k_ / n_, f'{k_}/{n_}', va='center', fontsize=7.5, color=INK)
    p1, p2 = (c1 == 'long').mean(), (c2 == 'long').mean(); chance = p1 * p2 + (1 - p1) * (1 - p2)
    ax.axhline(chance, color=INK2, ls='--', lw=1); ax.set_xlim(-0.6, 2.6); ax.text(-0.55, chance + 0.02, 'chance', fontsize=7, color=INK2)
    ax.set_xticks(range(3)); ax.set_xticklabels(qs); ax.set_xlabel('Tertile of between-window stability difference Δ')
    ax.set_ylabel('Same window chosen in both iterations'); ax.set_ylim(0, 1.05); ax.text(-0.18, 1.06, 'a', transform=ax.transAxes, fontsize=11, fontweight='bold')
    ax = axs[1]; rng = np.random.default_rng(0)
    colors = {'48 h-stable': C48, '8-day-stable': C8, 'Convergent': CC}; mk = {'48 h-stable': 'o', '8-day-stable': '^', 'Convergent': 's'}
    for i, m in enumerate([agree, ~agree]):
        v = dl[m.values]; cc = cls.reindex(v.index); xs = i + rng.uniform(-0.17, 0.17, len(v))
        for k_ in colors:
            sel = (cc == k_).values
            ax.scatter(xs[sel], v.values[sel], s=18, color=colors[k_], marker=mk[k_], edgecolors='white', linewidths=0.4, label=k_ if i == 0 else None)
        ax.plot([i - 0.27, i + 0.27], [v.median()] * 2, color=INK, lw=2); ax.text(i + 0.3, v.median(), f'{v.median():.3f}', va='center', fontsize=7.5)
    ax.set_xticks([0, 1]); ax.set_xticklabels([f'identical choice\n(n = {agree.sum()})', f'different choice\n(n = {(~agree).sum()})']); ax.set_xlim(-0.5, 1.6)
    p = R['window_choice']['mannwhitney_p']; mant, ex = f'{p:.1e}'.split('e')
    ax.set_ylabel('Stability difference Δ'); ax.set_title(rf'Mann–Whitney U test, p = {mant} $\times$ 10$^{{{int(ex)}}}$', fontsize=8, color=INK2, loc='right')
    ax.legend(frameon=False, fontsize=7, loc='upper right', bbox_to_anchor=(1.02, 0.9)); ax.text(-0.18, 1.06, 'b', transform=ax.transAxes, fontsize=11, fontweight='bold')
    fig.tight_layout(); fig.savefig(os.path.join(out, 'Fig6_window_choice.png'), dpi=300); fig.savefig(os.path.join(out, 'Fig6_window_choice.pdf')); plt.close(fig)
    # Supplementary Fig. S1: seeds and grid by metric
    fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.1)); ax = axs[0]
    for i, (s, l, c) in enumerate([('short', '48 h', C48), ('long', '8 days', C8), ('comb', 'Combined', CC)]):
        v = seeds_df[seeds_df.set == s].sil.values
        ax.scatter(i + rng.uniform(-0.18, 0.18, len(v)), v, s=10, color=c, alpha=0.55, edgecolors='none')
        ax.plot([i - 0.28, i + 0.28], [np.median(v)] * 2, color=INK, lw=2); ax.text(i + 0.3, np.median(v), f'{np.median(v):.3f}', va='center', fontsize=7.5)
    ax.set_xticks(range(3)); ax.set_xticklabels(['48 h', '8 days', 'Combined']); ax.set_ylabel('Silhouette score'); ax.set_xlim(-0.5, 2.8)
    ax.set_title(f'{seeds_df.seed.nunique()} Leiden seeds (line = median)', fontsize=8.5, loc='left', color=INK); panel(ax, 'a')
    if G is not None:
        ax = axs[1]
        W = G.pivot_table(index=['n_comps', 'metric', 'k', 'res'], columns='set', values=['sil', 'disc', 'ncl']).dropna(); W = W[W.ncl.min(1) >= 2]
        dS = W.sil.comb - W.sil[['short', 'long']].max(1); dD = W.disc.comb - W.disc[['short', 'long']].min(1)
        mets = [('correlation', 'correlation'), ('cosine', 'cosine'), ('euclidean', 'Euclidean')]
        for off, vals, col, lab_ in [(-0.12, dS, C48, 'higher silhouette than both windows'), (0.12, dD, C8, 'fewer discordant than the better window')]:
            for i, (m, _) in enumerate(mets):
                v = vals.xs(m, level='metric'); k_ = int((v > 0).sum() if col == C48 else (v < 0).sum()); n_ = len(v)
                lo, hi = proportion_confint(k_, n_, method='wilson'); pct = 100 * k_ / n_
                ax.plot([i + off] * 2, [100 * lo, 100 * hi], color=INK2, lw=1.2, zorder=2)
                ax.scatter([i + off], [pct], s=55, color=col, edgecolors=INK, linewidths=0.8, zorder=3, label=lab_ if i == 0 else None)
                ax.text(i + off + 0.07, pct, f'{pct:.0f}', va='center', fontsize=7.5, color=INK)
        ax.set_xticks(range(3)); ax.set_xticklabels([m[1] for m in mets]); ax.set_ylabel('Configurations where\nCombined is better (%)')
        ax.set_ylim(0, 112); ax.set_xlim(-0.5, 2.6); ax.axhline(50, color=GRAY, lw=1.0, ls='--')
        ax.legend(frameon=False, fontsize=7, loc='upper right', bbox_to_anchor=(1.02, 1.12)); ax.set_xlabel('Distance metric'); panel(ax, 'b')
    fig.tight_layout(); fig.savefig(os.path.join(out, 'FigS1_seeds_grid.png'), dpi=300); fig.savefig(os.path.join(out, 'FigS1_seeds_grid.pdf')); plt.close(fig)

## Run

In [ ]:
CONFIG['FILE'] = 'INPUT'   # path to the per-well CSV table
CONFIG['OUT_DIR'] = 'Result'
results = main(CONFIG)